In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector

In [28]:
columns = [
    "class",
    "cap-shape", "cap surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root",
    "stalk-surface-above-ring", "stalk-surface-below-ring",
    "stalk-color-above-ring", "stalk-color-below-ring",
    "veil-type", "veil-color",
    "ring-number", "ring-type",
    "spore-print-color",
    "population", "habitat"
]

df = pd.read_csv("agaricus-lepiota.data", names=columns)

print("Original shape:", df.shape)

df = df.replace("?", pd.NA)
df = df.dropna()

print("After cleaning:", df.shape)
df.head()


Original shape: (8124, 23)
After cleaning: (5644, 23)


,class,cap-shape,cap surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [29]:
y = df["class"]
X = df.drop(columns=["class"])

for col in X.columns:
  X[col] = LabelEncoder().fit_transform(X[col])
y= LabelEncoder().fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

feature_names = X.columns.to_numpy()

print("Data ready:", X_scaled.shape)



Data ready: (5644, 22)


In [30]:
from sklearn.model_selection import RepeatedKFold
rkf = RepeatedKFold(n_splits=10, n_repeats=10, random_state=42)
models = {
    "Linear Classifier": Perceptron(max_iter=1000, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Gaussian NB": GaussianNB(),
    "Neural Network": MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42)
}

f = RepeatedKFold(n_splits=10, n_repeats=10, random_state=42)

part2_results = []

print("Running Part 2")

for name, model in models.items():
  scores = cross_val_score(
      model,
      X_scaled,
      y,
      cv=rkf,
      scoring="accuracy",
      n_jobs=-1
  )

  part2_results.append([name, scores.mean(), scores.std()])
  print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")

  part2_df = pd.DataFrame(
      part2_results,
      columns=["Algorithm", "Mean Accuracy", "Std" ]
  )
  part2_df

Running Part 2
Linear Classifier: mean=0.9857, std=0.0166
Logistic Regression: mean=0.9887, std=0.0045
KNN: mean=1.0000, std=0.0000
Gaussian NB: mean=0.6277, std=0.0176
Neural Network: mean=1.0000, std=0.0000


In [36]:
fs_cv = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42)
part3_results = []
print("Running Part 3 Feature Selection")
for name, model in models.items():
  print(f"\n{name}")
  sfs = SequentialFeatureSelector(
      estimator=model,
      n_features_to_select=3,
      direction="forward",
      cv=fs_cv,
      n_jobs=-1
  )
  sfs.fit(X_scaled,y)
  selected_features = feature_names[sfs.get_support()]
  X_selected = X_scaled[:, sfs.get_support()]
  scores = cross_val_score(
      model,
      X_selected,
      y,
      cv=fs_cv,
      scoring="accuracy",
      n_jobs=-1
  )

  part3_results.append([
      name,
      ", ".join(selected_features),
      scores.mean(),
      scores.std()
  ])

  print("Selected:", list(selected_features))
  print(f"mean={scores.mean():.4f}, std={scores.std():.4f}")

part3_df = pd.DataFrame(
    part3_results,
    columns=["Algorithm", "Best Feature Subset", "Mean Accuracy", "Std"]
)
part3_df

Running Part 3 Feature Selection

Linear Classifier
Selected: ['bruises', 'stalk-shape', 'spore-print-color']
mean=0.8554, std=0.0540

Logistic Regression
Selected: ['gill-size', 'ring-number', 'spore-print-color']
mean=0.9584, std=0.0090

KNN
Selected: ['cap-color', 'odor', 'spore-print-color']
mean=1.0000, std=0.0000

Gaussian NB
Selected: ['gill-spacing', 'gill-size', 'spore-print-color']
mean=0.9440, std=0.0060

Neural Network
Selected: ['cap-color', 'odor', 'spore-print-color']
mean=1.0000, std=0.0000


,Algorithm,Best Feature Subset,Mean Accuracy,Std
0,Linear Classifier,"bruises, stalk-shape, spore-print-color",0.855430,0.053983
1,Logistic Regression,"gill-size, ring-number, spore-print-color",0.958365,0.008957
2,KNN,"cap-color, odor, spore-print-color",1.000000,0.000000
3,Gaussian NB,"gill-spacing, gill-size, spore-print-color",0.944012,0.006012
4,Neural Network,"cap-color, odor, spore-print-color",1.000000,0.000000
